# 🏠 California Housing Price Prediction

A complete ML pipeline: EDA → Preprocessing → Model Comparison → Hyperparameter Tuning

**Best Result: Random Forest — R² = 0.805**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries loaded ✅')

## 1. Load Dataset

In [ ]:
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['Price'] = housing.target

print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('Basic Statistics:')
display(df.describe())
print(f'\nMissing values: {df.isnull().sum().sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Price distribution
sns.histplot(df['Price'], kde=True, ax=axes[0])
axes[0].set_title('House Price Distribution')
axes[0].set_xlabel('Price (100k USD)')

# Correlation heatmap
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', ax=axes[1])
axes[1].set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.show()

In [ ]:
# Geographic distribution
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    df['Longitude'], df['Latitude'],
    alpha=0.4, s=df['Population']/50,
    c=df['Price'], cmap='viridis'
)
plt.colorbar(scatter, label='Price (100k USD)')
plt.title('California House Prices — Geographic Distribution')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

## 3. Preprocessing

In [ ]:
# Outlier capping (IQR method)
df_clean = df.copy()
for col in df_clean.select_dtypes(include=[np.number]).columns:
    Q1, Q3 = df_clean[col].quantile(0.25), df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    df_clean[col] = np.clip(df_clean[col], Q1 - 1.5*IQR, Q3 + 1.5*IQR)
print('Outliers capped ✅')

In [ ]:
# Feature Engineering
df_clean['RoomsPerHousehold'] = df_clean['AveRooms']  / df_clean['AveOccup']
df_clean['BedroomRatio']      = df_clean['AveBedrms'] / df_clean['AveRooms']
df_clean['PopulationDensity'] = df_clean['Population']/ df_clean['AveOccup']

print('New features added:')
print('  RoomsPerHousehold, BedroomRatio, PopulationDensity ✅')
print(f'Total features: {df_clean.shape[1] - 1}')

In [ ]:
# Split, Scale, Train/Test
X = df_clean.drop('Price', axis=1)
y = df_clean['Price']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape} ✅')

## 4. Model Training & Comparison

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, name):
    t0 = time.time()
    model.fit(X_train, y_train)
    t = time.time() - t0
    y_pred = model.predict(X_test)
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    return {
        'Model': name,
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'Test MAE':  mean_absolute_error(y_test, y_pred),
        'Test R²':   r2_score(y_test, y_pred),
        'CV RMSE':   np.sqrt(-cv.mean()),
        'Time(s)':   round(t, 2),
        'model':     model,
        'y_pred':    y_pred
    }

models = {
    'Linear Regression':  LinearRegression(),
    'Ridge Regression':   Ridge(alpha=1.0),
    'Lasso Regression':   Lasso(alpha=0.1),
    'Random Forest':      RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = [evaluate_model(m, X_train, X_test, y_train, y_test, n) for n, m in models.items()]
results_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['model','y_pred']} for r in results])
results_df = results_df.sort_values('Test RMSE').reset_index(drop=True)
display(results_df)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ['Test RMSE', 'Test R²', 'Test MAE']):
    bars = ax.bar(results_df['Model'], results_df[metric],
                  color=sns.color_palette('viridis', len(results_df)))
    ax.set_title(metric, fontsize=13)
    ax.set_xticklabels(results_df['Model'], rotation=30, ha='right')
plt.suptitle('Model Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Feature Importance (Best Model: Random Forest)

In [ ]:
best_model = next(r['model'] for r in results if r['Model'] == 'Random Forest')
feature_names = X.columns.tolist()
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))
plt.bar(range(len(feature_names)), importances[indices],
        color=sns.color_palette('viridis', len(feature_names)))
plt.xticks(range(len(feature_names)), [feature_names[i] for i in indices], rotation=45, ha='right')
plt.title('Random Forest — Feature Importances', fontsize=14)
plt.xlabel('Features')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

## 6. Actual vs Predicted (Best Model)

In [ ]:
best_result = next(r for r in results if r['Model'] == 'Random Forest')
y_pred_best = best_result['y_pred']

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_best, alpha=0.4, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title(f'Random Forest — Actual vs Predicted  |  R² = {best_result["Test R²"]:.4f}')
plt.tight_layout()
plt.show()

## 7. Summary

| | |
|---|---|
| **Best Model** | Random Forest |
| **Test R²** | 0.8053 |
| **Test RMSE** | 0.4962 |
| **Test MAE** | 0.3284 |

**Key takeaways:**
- Tree-based models (RF, GBM) significantly outperform linear models on this dataset
- `MedInc` (median income) is the strongest predictor of house price
- Feature engineering added meaningful signals (especially `RoomsPerHousehold`)
- Linear models struggle because the price-income relationship is non-linear

📂 Full code: [GitHub](https://github.com/YOUR_USERNAME/california-housing-price-prediction)